# Embedding
  
Das Notebook dient dazu per:
  
- **CLIP**          CLIP -> 
- **AnyLoc**        DINOv2 -> Feature Aggregation -> Descriptor -> Retrival (Github: https://github.com/AnyLoc/Revisit-Anything.git)
- **EigenPlaces**   Backbone -> VPR-Descriptor -> Retrival (Github: https://github.com/gmberton/EigenPlaces.git)
- **MixVPR**        noch keine Idee (Mixed ansatz)
  
die Bilder in Vectorinformationen zu embedden


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
import sys



plt.rcParams["figure.dpi"] = 300

def find_project_root():
    for dir in (Path.cwd(), *Path.cwd().parents):
        if (dir / "config.yaml").exists():
            return dir
    raise FileNotFoundError("Projektroot nicht gefunden")


def find_upwards(name):
    # Erlaubt, das Notebook aus eval/ oder aus dem Projektwurzelverzeichnis
    # zu starten, ohne Pfade anzupassen.
    for d in [Path.cwd(), *Path.cwd().parents]:
        if (d / name).exists():
            return d / name
    return None


PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models.adapter import LinearAdapter


CFG_FILE = find_upwards("config.yaml")
assert CFG_FILE, "config.yaml nicht gefunden (liegt im Projektwurzelverzeichnis)."
CFG = yaml.safe_load(CFG_FILE.read_text())

METHOD = CFG["vpr"]["method"]
MODEL_ID = CFG["vpr"]["models"][METHOD]
ADAPTER = CFG["vpr"].get("adapter", "none")


N_IMAGES = None
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
IMAGE_PATH = Path.home() / "Downloads" / "images"
EMBEDDING_DIR = PROJECT_ROOT / "data" / "embeddings"
EMBEDDING_DIR.mkdir(parents=True, exist_ok=True)
METHOD_DIR = EMBEDDING_DIR / f"{METHOD}"
METHOD_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH_META = PROCESSED_DIR / "metadata.parquet"

if ADAPTER == "none" or ADAPTER == "None":
    EMBEDDING_NAME = METHOD
else:
    EMBEDDING_NAME = f"{METHOD}_{ADAPTER}"


metadata = pd.read_parquet(DATA_PATH_META)
embedding_metadata = metadata[metadata["split"].isin(["database", "query"])].copy()
embedding_metadata = embedding_metadata.reset_index(drop=True)
embedding_path = METHOD_DIR / f"{EMBEDDING_NAME}_embeddings.npy"
metadata_path = METHOD_DIR / f"{EMBEDDING_NAME}_metadata.parquet"


# A GPU makes the image encoder roughly 20x faster, but nothing here *needs* one.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 64
count = len(list(IMAGE_PATH.glob("*.jpg"))) + len(list(IMAGE_PATH.glob("*.png")))


# Wieviel sollen Embedded werden 
if N_IMAGES is not None:

    if N_IMAGES <= 0:
        raise ValueError("N_IMAGES muss None oder größer Null sein")
    
    if N_IMAGES > len(embedding_metadata):
        raise ValueError(
            f"N_IMAGES darf nicht größer als {len(embedding_metadata)} sein ist aber {N_IMAGES} "
        )
    embedding_metadata = embedding_metadata.iloc[:N_IMAGES].copy()


image_paths = [ IMAGE_PATH / f"{image_id}.jpg" for image_id in embedding_metadata["image_id"]]


print(f"Bilder:             {len(metadata):,}")
print(f"Bilder downloaded:  {count}")
print(f"Bildordner:         {IMAGE_PATH}")
print(f"Embedding-Ordner:   {EMBEDDING_DIR}")
print(f"running on:         {DEVICE}")
print(f"batch size:         {BATCH_SIZE}")
print(f"Method:             {METHOD}")
print(f"MODEL_ID:           {MODEL_ID}")


# Modell laden
  
**MODEL_REVISION** = "3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268" for _reloading_ Modell



In [ ]:
MODEL_REVISION = "3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268"

if METHOD == "clip":
    from src.models.clip import CLIPEmbedder
    embedder = CLIPEmbedder(model_id = MODEL_ID, device = DEVICE, revision = MODEL_REVISION, num_workers=8, use_amp=True)
elif METHOD == "anyloc":
    from src.models.anyloc import AnyLocEmbedder
    embedder = AnyLocEmbedder(model_id = MODEL_ID, device = DEVICE, revision = MODEL_REVISION)
elif METHOD == "eigenplaces":
    from src.models.eigenplaces import EigenPlacesEmbedder
    embedder = EigenPlacesEmbedder(model_id=MODEL_ID, device=DEVICE, revision=MODEL_REVISION)
elif METHOD == "mixvpr":
    from src.models.mixvpr import MixEmbedder
    embedder = MixEmbedder(model_id=MODEL_ID, device=DEVICE, revision=MODEL_REVISION)
else:
    raise ValueError(f"Unbekannte METHOD = {METHOD}")


EMBEDDING_DIM = embedder.embedding_dim

adapter = None

if ADAPTER == "linear":
    adapter_path = (PROJECT_ROOT / "models" / "adapters" / f"{METHOD}_linear.pt")
    if not adapter_path.exists():
        raise FileNotFoundError(f"Adapter wurde nicht gefunden unter: {adapter_path}")

    adapter = LinearAdapter(embedding_dim = EMBEDDING_DIM).to(DEVICE).eval()
    adapter.load_state_dict(torch.load(adapter_path, map_location = DEVICE))
    adapter.eval()
elif ADAPTER in ("none", "None"):
    adapter = None

else:
    raise ValueError(f"Unbekaannter ADAPTER: {ADAPTER}")


print(f"adapter:            {ADAPTER}")
print(f"Embedding dimension: {EMBEDDING_DIM}")


# Embedding Creation

In [ ]:


embeddings = embedder.embed_images(image_paths, batch_size=BATCH_SIZE)

print("Embedding shape:", embeddings.shape)
print("Embedding dtype:", embeddings.dtype)
print("Erstes Embedding:", embeddings[0][:10])
print("NaN:", np.isnan(embeddings).any())
print("Inf:", np.isinf(embeddings).any())

if adapter is not None:
    with torch.inference_mode():
        embeddings = adapter(torch.from_numpy(embeddings).float().to(DEVICE))

        embeddings = embeddings.cpu().numpy()

    embeddings = embeddings / np.linalg.norm(embeddings, axis = 1, keepdims = True)


# Nur möglich wenn alle embedded wurden
# assert len(embeddings) == len(embedding_metadata)
assert embeddings.shape[1] == EMBEDDING_DIM
assert np.isfinite(embeddings).all()
norms = np.linalg.norm(embeddings, axis=1)



print(f"Shape:                 {embeddings.shape}")
print(f"Dtype:                 {embeddings.dtype}")
print(f"Normalized Minimum:    {norms.min():.2f}")
print(f"Normalized Maximum:    {norms.max():.2f}")
print(f"Normalized Mittelwert: {norms.mean():.2f}")


# Embedding Speichern


In [ ]:
np.save( embedding_path, embeddings)
embedding_metadata.to_parquet(metadata_path, index = False)

print(f"Embeddings gespeichert in:  {embedding_path}")
print(f"Metadaten gespeichert in:   {metadata_path}")